# City Dimension Table

This step builds and maintains the `workspace.open_meteo.dim_city` table incrementally on each batch run:

1. **Geocode cities** — resolve each configured city (name + country) into geographic metadata (id, coordinates, timezone, elevation). Cities with no match are logged and skipped.
2. **Create if missing** — `CREATE TABLE IF NOT EXISTS` provisions the dimension table on the first run; later runs reuse it.
3. **Incremental merge** — geocoded rows are staged in a temp view and applied with a `MERGE` keyed on `id`: existing rows are updated, new cities are inserted. Re-running is safe and will not duplicate rows.


In [0]:
from open_meteo.client import City, geocode_cities
import logging

log = logging.getLogger("open_meteo.dim_city")


CITY_DIM_TABLE = "workspace.open_meteo.dim_city"
CITY_COLUMNS = ", ".join(
    (
        "id BIGINT",
        "country_id INT",
        "name STRING",
        "country STRING",
        "country_code STRING",
        "timezone STRING",
        "latitude DOUBLE",
        "longitude DOUBLE",
        "elevation DOUBLE",
    )
)

geocode = await geocode_cities(
    [
        City("London", "GB"),
        City("Paris", "FR"),
        City("Rome", "IT"),
        City("Madrid", "ES"),
        City("Berlin", "DE"),
        City("Moscow", "RU"),
        City("New York", "US"),
    ]
)

missing = [city for city, match in geocode.items() if match is None]
if len(missing) > 0:
    log.warning("No geocoding match for %s", missing)

rows = [m.model_dump() for m in geocode.values() if m is not None]

spark.sql(f"CREATE TABLE IF NOT EXISTS {CITY_DIM_TABLE} ({CITY_COLUMNS})")

spark.createDataFrame(rows, CITY_COLUMNS).createOrReplaceTempView("city_staging")

spark.sql(f"""
    MERGE INTO {CITY_DIM_TABLE} AS t
    USING city_staging AS s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")